En este archivo se realiza extracción de datos de Eurostat, calcula el PIB per cápita y prepara el terreno para el cruce con la European Social Survey (ESS).

In [1]:
pip install pandas numpy eurostat pyreadstat matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# IMPORTACIÓN DE LIBRERÍAS
import pandas as pd
import numpy as np
import eurostat
import os  # Importación obligatoria al inicio para evitar NameError
import sys
import pyreadstat  # Necesaria para leer archivos .sav de la ESS más adelante
import matplotlib.pyplot as plt
print("Librerías importadas correctamente.")

Librerías importadas correctamente.


In [3]:
# Vamos a descargar el PIB (nama_10_gdp) y la Población (demo_pjan)
print("Descargando datos de Eurostat (esto puede tardar unos segundos)...")

# PIB a precios corrientes
df_gdp_raw = eurostat.get_data_df('nama_10_gdp')
# Población a 1 de enero
df_pop_raw = eurostat.get_data_df('demo_pjan')

print("Datos descargados.")

Descargando datos de Eurostat (esto puede tardar unos segundos)...
Datos descargados.


In [4]:
# LIMPIEZA Y FILTRADO INICIAL
# Eurostat usa nombres de columnas compuestos (ej. 'geo\\time'). 
# Esta función limpia los nombres para quedarnos solo con la parte importante.
def clean_columns(df):
    # Dividimos por la barra invertida y nos quedamos con la primera parte (ej. 'geo')
    df.columns = [str(col).split('\\')[0] for col in df.columns]
    # Por si acaso el año viene como número, lo pasamos a string para evitar errores
    df.columns = [str(col) for col in df.columns]
    return df

df_gdp = clean_columns(df_gdp_raw)
df_pop = clean_columns(df_pop_raw)

# Elegimos el año más reciente con datos consolidados
ANIO = '2022'

# Filtramos PIB: Precios corrientes (CP_MEUR) e indicador PIB (B1GQ)
# Añadimos drop_duplicates para evitar que versiones con distinto ajuste estacional dupliquen filas
pib_data = df_gdp[(df_gdp['unit'] == 'CP_MEUR') & (df_gdp['na_item'] == 'B1GQ')][['geo', ANIO]]
pib_data = pib_data.drop_duplicates(subset=['geo']).rename(columns={ANIO: 'pib_millones'})

# Filtramos Población: Solo el total (age TOTAL) y total de sexos (sex T) para evitar duplicados
pop_data = df_pop[(df_pop['age'] == 'TOTAL') & (df_pop['sex'] == 'T')][['geo', ANIO]]
pop_2022 = pop_data.drop_duplicates(subset=['geo']).rename(columns={ANIO: 'poblacion'})

In [ ]:
# CÁLCULO DE MEDIDA PROPIA: PIB PER CÁPITA
# Unimos ambos datasets por la columna 'geo'
df_final = pd.merge(pib_data, pop_2022, on='geo').dropna()

# Calculamos: (PIB en millones * 1.000.000) / Población
df_final['pib_per_capita'] = (df_final['pib_millones'].astype(float) * 1000000) / df_final['poblacion'].astype(float)

# Definimos los países que pertenecen a la Unión Europea (códigos ISO-2)
paises_ue = [
    'AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR', 
    'DE', 'GR', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL', 
    'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE'
]

# Filtramos solo UE y obtenemos el TOP 10 de forma limpia
top_paises = df_final[df_final['geo'].isin(paises_ue)].sort_values('pib_per_capita', ascending=False)

print("\n--- RANKING PAÍSES UE POR PIB PER CÁPITA (Limpio) ---")
print(top_paises[['geo', 'pib_per_capita']].reset_index(drop=True))


--- TOP 10 PAÍSES UE POR PIB PER CÁPITA (Limpio) ---
   geo  pib_per_capita
0   LU   118889.923566
1   IE   101026.467922
2   DK    64794.855468
3   NL    56496.988859
4   SE    52351.065208
5   AT    50048.530287
6   BE    48347.084425
7   FI    47967.454910
8   DE    47928.013467
9   FR    38976.807497
10  MT    34574.584658
11  IT    33848.349961
12  CY    31884.865407
13  ES    28973.562214
14  CZ    27287.705172
15  EE    27257.102439
16  SI    26994.181797
17  LT    23906.324951
18  PT    23409.880150
19  SK    20232.866065
20  LV    19239.539023
21  PL    17937.559964
22  HU    17537.890971
23  HR    17504.935524
24  RO    14744.313136
25  BG    13278.582716


In [6]:
# PREPARACIÓN PARA LA ESS
# Aquí guardamos la lista de nuestros 10 países objetivo
lista_paises_objetivo = top_paises['geo'].tolist()

# NOTA PARA EL FUTURO:
# Al descargar los archivos ESS9, ESS10 y ESS11:
# 1. Carga el archivo: ess_data, meta = pyreadstat.read_spss('archivo.sav')
# 2. Filtra: ess_top = ess_data[ess_data['cntry'].isin(lista_paises_objetivo)]
# 3. Recuerda usar ESS10SC para Alemania (DE) y Austria (AT) si no están en la R10 normal.

print("\nListo para el siguiente paso: Carga de microdatos de la ESS.")


Listo para el siguiente paso: Carga de microdatos de la ESS.


In [7]:
archivo_ess = 'ess_cumulative.sav'